In [0]:
import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp, col

print("1. Fetching AirLabs Secret...")
airlabs_key = dbutils.secrets.get(scope="aviation_api_keys", key="airlabs_key")
airlabs_url = "https://airlabs.co/api/v9/schedules"
all_valid_flights = []

In [0]:
# Part 2a: Live Departures (LIT)
print("2a. Fetching Live Departures from LIT...")
dep_params = {"api_key": airlabs_key, "dep_iata": "LIT"}
dep_res = requests.get(airlabs_url, params=dep_params)

if dep_res.status_code == 200:
    valid_dep_statuses = ['active', 'landed', 'diverted', 'cancelled']
    for f in dep_res.json().get("response", []):
        if f.get("status") in valid_dep_statuses:
            f['direction'] = 'departure'
            all_valid_flights.append(f)
    print(f"   -> Retained {len([f for f in all_valid_flights if f['direction'] == 'departure'])} active/completed departures.")
else:
    print(f"   -> [!] FAILED AirLabs Departures: {dep_res.text}")

In [0]:
# Part 2b: Live Arrivals (LIT)
print("2b. Fetching Live Arrivals to LIT...")
arr_params = {"api_key": airlabs_key, "arr_iata": "LIT"}
arr_res = requests.get(airlabs_url, params=arr_params)

if arr_res.status_code == 200:
    valid_arr_statuses = ['landed', 'diverted', 'cancelled']
    for f in arr_res.json().get("response", []):
        if f.get("status") in valid_arr_statuses:
            f['direction'] = 'arrival'
            all_valid_flights.append(f)
    print(f"   -> Retained {len([f for f in all_valid_flights if f['direction'] == 'arrival'])} completed arrivals.")
else:
    print(f"   -> [!] FAILED AirLabs Arrivals: {arr_res.text}")

In [0]:
if all_valid_flights:
    print("4. Writing valid flights to Bronze layer...")
    pdf_flights = pd.DataFrame(all_valid_flights)
    sdf_flights = spark.createDataFrame(pdf_flights)
    
    # THE STRING SHIELD: Force every API column to String to prevent Pandas/Delta merge crashes
    for c in sdf_flights.columns:
        sdf_flights = sdf_flights.withColumn(c, col(c).cast("string"))
            
    # Add the ingestion timestamp
    sdf_flights = sdf_flights.withColumn("bronze_ingested_at", current_timestamp())
    
    sdf_flights.write.format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("aviation_project.bronze_live_flights")
    print("   -> SUCCESS: Flights appended to aviation_project.bronze_live_flights.")
else:
    print("   -> No new valid flights met the status criteria in this window.")